# Online Retail Exploratory Data Analysis Project

This notebook analyses the Online Retail dataset from 2010–2011 to understand sales performance, customer activity, country performance, best-selling products, seasonality, and anomalies.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)

In [ ]:
df = pd.read_excel("Online Retail.xlsx")
df.head()

In [ ]:
df.info()
df.describe()

## Data Cleaning

The dataset contains cancelled invoices, negative quantities, zero/negative prices, and missing customer IDs. For sales analysis, I created a cleaned dataset containing only completed purchases with positive quantity, positive price, and known customers.

In [ ]:
df.isna().sum()

# Create a working copy
data = df.copy()

# Convert InvoiceNo to string so cancelled invoices can be identified
data["InvoiceNo"] = data["InvoiceNo"].astype(str)

# Check anomalies before cleaning
print("Cancelled invoices:", data["InvoiceNo"].str.startswith("C").sum())
print("Negative quantities:", (data["Quantity"] < 0).sum())
print("Zero prices:", (data["UnitPrice"] == 0).sum())
print("Negative prices:", (data["UnitPrice"] < 0).sum())

# Keep completed sales only
retail = data[
    (~data["InvoiceNo"].str.startswith("C")) &
    (data["Quantity"] > 0) &
    (data["UnitPrice"] > 0) &
    (data["CustomerID"].notna())
].copy()

# Create sales value column
retail["TotalSales"] = retail["Quantity"] * retail["UnitPrice"]

# Create date features
retail["InvoiceDate"] = pd.to_datetime(retail["InvoiceDate"])
retail["YearMonth"] = retail["InvoiceDate"].dt.to_period("M").astype(str)
retail["DayOfWeek"] = retail["InvoiceDate"].dt.day_name()
retail["Hour"] = retail["InvoiceDate"].dt.hour

retail.head()

## Summary of Cleaned Data

In [ ]:
print("Original rows:", len(df))
print("Rows after cleaning:", len(retail))
print("Total sales: £{:,.2f}".format(retail["TotalSales"].sum()))
print("Total quantity sold: {:,.0f}".format(retail["Quantity"].sum()))
print("Unique invoices:", retail["InvoiceNo"].nunique())
print("Unique customers:", retail["CustomerID"].nunique())
print("Countries:", retail["Country"].nunique())

## Basic Statistics

In [ ]:
retail[["Quantity", "UnitPrice", "TotalSales"]].describe()

## Sales Trends Over Time

In [ ]:
monthly_sales = retail.groupby("YearMonth").agg(
    total_sales=("TotalSales", "sum"),
    total_quantity=("Quantity", "sum"),
    orders=("InvoiceNo", "nunique")
).reset_index()

monthly_sales

In [ ]:
plt.figure(figsize=(12, 5))
sns.lineplot(data=monthly_sales, x="YearMonth", y="total_sales", marker="o")
plt.title("Monthly Sales Trend")
plt.xlabel("Month")
plt.ylabel("Total Sales (£)")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## Busiest Days of the Week

In [ ]:
day_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
weekly_sales = retail.groupby("DayOfWeek").agg(
    total_sales=("TotalSales", "sum"),
    total_quantity=("Quantity", "sum"),
    orders=("InvoiceNo", "nunique")
).reindex(day_order).reset_index()

weekly_sales

In [ ]:
plt.figure(figsize=(10, 5))
sns.barplot(data=weekly_sales, x="DayOfWeek", y="total_sales")
plt.title("Sales by Day of Week")
plt.xlabel("Day")
plt.ylabel("Total Sales (£)")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## Top-Selling Products

In [ ]:
top_products = retail.groupby(["StockCode", "Description"]).agg(
    total_quantity=("Quantity", "sum"),
    total_sales=("TotalSales", "sum")
).reset_index().sort_values("total_quantity", ascending=False).head(10)

top_products

In [ ]:
plt.figure(figsize=(12, 6))
sns.barplot(data=top_products, y="Description", x="total_quantity")
plt.title("Top 10 Products by Quantity Sold")
plt.xlabel("Quantity Sold")
plt.ylabel("Product")
plt.tight_layout()
plt.show()

## Top Countries

In [ ]:
top_countries = retail.groupby("Country").agg(
    total_quantity=("Quantity", "sum"),
    total_sales=("TotalSales", "sum"),
    orders=("InvoiceNo", "nunique"),
    customers=("CustomerID", "nunique")
).reset_index().sort_values("total_sales", ascending=False).head(10)

top_countries

In [ ]:
plt.figure(figsize=(12, 6))
sns.barplot(data=top_countries, y="Country", x="total_sales")
plt.title("Top 10 Countries by Total Sales")
plt.xlabel("Total Sales (£)")
plt.ylabel("Country")
plt.tight_layout()
plt.show()

## Outlier Detection

In [ ]:
plt.figure(figsize=(10, 4))
sns.boxplot(x=retail["TotalSales"])
plt.title("Outliers in Transaction Sales Value")
plt.xlabel("Total Sales (£)")
plt.tight_layout()
plt.show()

retail.sort_values("TotalSales", ascending=False).head(10)[["InvoiceNo", "StockCode", "Description", "Quantity", "UnitPrice", "TotalSales", "Country"]]

## Key Findings

- The cleaned dataset contains 397,884 valid sales rows from 541,909 original rows.
- Total cleaned sales were approximately £8.91 million.
- The United Kingdom generated the largest share of sales by a very large margin.
- The busiest sales months were November, October, and September 2011, showing a strong seasonal increase before Christmas.
- Thursday produced the highest sales by day of week.
- The data contains major outliers, especially very large quantity purchases. These can strongly affect averages, so medians and visual checks are important.
- Missing CustomerID values were removed for customer-level analysis, but they could still be useful for general transaction-level analysis if customer identity is not required.

## Recommendations

1. Prepare inventory earlier for the September–November peak season.
2. Focus marketing and stock planning around the best-selling products.
3. Investigate high-value countries outside the UK, especially Netherlands, EIRE, Germany, and France.
4. Review extreme orders separately before making business decisions because they may represent wholesale purchases, data entry errors, or unusual one-off transactions.
5. Improve customer data capture because many rows are missing CustomerID, limiting customer segmentation and retention analysis.